<a href="https://colab.research.google.com/github/otitamario/sp-pa-gep/blob/main/experiments/Experiment_Image_Recovering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Clone the repository into Colab runtime
!git clone https://github.com/otitamario/sp-pa-gep.git

# Move into repo root
%cd sp-pa-gep

# Make sure Python sees the project root
import sys
sys.path.append(".")

Cloning into 'sp-pa-gep'...
remote: Enumerating objects: 309, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 309 (delta 79), reused 23 (delta 23), pack-reused 184 (from 1)
Receiving objects: 100% (309/309), 5.86 MiB | 31.41 MiB/s, done.
Resolving deltas: 100% (147/147), done.
/content/sp-pa-gep


In [2]:
import os
# =========================
# OUTPUT DIRECTORY (save plots here)
# =========================
FIGDIR = "figures"
os.makedirs(FIGDIR, exist_ok=True)

In [20]:
import os
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List
from skimage import data
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from scipy.ndimage import gaussian_filter

# -----------------------------
# Image restoration problem data
# -----------------------------
np.random.seed(0)

# Load standard public test image
img = data.camera().astype(np.float64)

# Optional downsampling for speed: 512x512 -> 256x256
#img = img[::2, ::2]
#img = img[::8, ::8]   # 64x64
img = img[::4, ::4]   # 128x128

M, N = img.shape
D = M * N

# Ground-truth vector
x_true = img.reshape(-1)

# -----------------------------
# Blur operator A and adjoint A*
# -----------------------------
BLUR_SIGMA = 1.0
NOISE_STD = 2.0

def A(x):
    X = x.reshape(M, N)
    Y = gaussian_filter(X, sigma=BLUR_SIGMA, mode="reflect")
    return Y.reshape(-1)

def AT(x):
    # For symmetric Gaussian blur with reflect boundary handling,
    # we use the same operator as adjoint numerically.
    X = x.reshape(M, N)
    Y = gaussian_filter(X, sigma=BLUR_SIGMA, mode="reflect")
    return Y.reshape(-1)

# Observed degraded image
noise = NOISE_STD * np.random.randn(D)
y_obs = A(x_true) + noise

# -----------------------------
# Feasible set C = [0,255]^D
# -----------------------------
def proj_C(x):
    return np.clip(x, 0.0, 255.0)

def constraint_violation(x):
    below = np.maximum(0.0, -x)
    above = np.maximum(0.0, x - 255.0)
    return float(np.max(np.maximum(below, above)))

# -----------------------------
# Objective and gradient
# -----------------------------
def phi(x):
    r = A(x) - y_obs
    return 0.5 * np.dot(r, r)

def grad_phi(x):
    return AT(A(x) - y_obs)

# -----------------------------
# Inner solver for resolvent
# -----------------------------
def estimate_lipschitz(num_power_iter=20):
    """
    Estimate ||A^*A|| by power iteration.
    Then L = ||A^*A|| + 1/r for the resolvent subproblem gradient.
    """
    z = np.random.randn(D)
    z /= np.linalg.norm(z)

    for _ in range(num_power_iter):
        z = AT(A(z))
        nz = np.linalg.norm(z)
        if nz == 0:
            return 1.0
        z /= nz

    Az = AT(A(z))
    return float(np.dot(z, Az))

ATA_norm_est = estimate_lipschitz()

def resolvent_image(x_n, r=1.0, inner_it=200, inner_tol=1e-4):
    """
    Solve approximately:
        u_n = argmin_{y in C} 0.5 ||A(y)-y_obs||^2 + (1/(2r)) ||y-x_n||^2

    by projected gradient.
    """
    y = proj_C(x_n.copy())

    L = ATA_norm_est + 1.0 / r
    step = 1.0 / L

    converged = False
    for j in range(inner_it):
        grad = grad_phi(y) + (1.0 / r) * (y - x_n)
        y_new = proj_C(y - step * grad)

        if np.linalg.norm(y_new - y) <= inner_tol:
            y = y_new
            converged = True
            return y, (j + 1), False, converged

        y = y_new

    return y, inner_it, True, converged

In [21]:
import src.benchkit as bk
import time

r = 1.0
maxit = 250
inner_it = 50
stop_tol = 1e-4

def make_sppa_step():
    def step_fn(x, k):
        alpha = 1.0 / (k + 2.0)

        u, inner_iters, inner_hit_max, ok = resolvent_image(
            x, r=r, inner_it=inner_it, inner_tol=1e-4
        )

        residual = float(np.linalg.norm(x - u))
        x_new = alpha * u_anchor + (1.0 - alpha) * u

        return x_new, {
            "x_new": x_new,
            "u": u,
            "residual": residual,
            "inner_iters": inner_iters,
            "inner_hit_max": inner_hit_max,
            "ok": ok,
            "constraint_violation": constraint_violation(x_new),
        }
    return step_fn


def make_wppa_step():
    def step_fn(x, k):
        alpha = 1.0 / (k + 2.0)

        u, inner_iters, inner_hit_max, ok = resolvent_image(
            x, r=r, inner_it=inner_it, inner_tol=1e-4
        )

        residual = float(np.linalg.norm(x - u))
        x_new = alpha * x + (1.0 - alpha) * u

        return x_new, {
            "x_new": x_new,
            "u": u,
            "residual": residual,
            "inner_iters": inner_iters,
            "inner_hit_max": inner_hit_max,
            "ok": ok,
            "constraint_violation": constraint_violation(x_new),
        }
    return step_fn

In [22]:
def residual_fn(x):
    return float(np.linalg.norm(x - proj_C(x - grad_phi(x))))

def error_fn(x):
    return float(np.linalg.norm(x - x_true))

x0 = proj_C(y_obs.copy())
u_anchor = x0.copy()   # or use x0.copy() if you prefer

logs_sppa, sum_sppa = bk.run(
    method_name="SPPA",
    x0=x0,
    max_iter=maxit,
    stop_tol_step=stop_tol,
    step_fn=make_sppa_step(),
    residual_fn=residual_fn,
    error_fn=error_fn,
)

logs_wppa, sum_wppa = bk.run(
    method_name="WPPA",
    x0=x0,
    max_iter=maxit,
    stop_tol_step=stop_tol,
    step_fn=make_wppa_step(),
    residual_fn=residual_fn,
    error_fn=error_fn,
)

bk.make_standard_plots(
    logs_by_method={"SPPA": logs_sppa, "WPPA": logs_wppa},
    outdir="figures",
    tag="image_gep",
    plot_residual=True,
    plot_error=True,
)

print(bk.latex_tables_split(
    [sum_sppa, sum_wppa],
    caption_perf="Computational performance for the image restoration experiment.",
    label_perf="tab:image_gep_perf",
    caption_acc="Accuracy and feasibility metrics for the image restoration experiment.",
    label_acc="tab:image_gep_acc",
))

\begin{table}[!ht]
\centering
\begin{tabular}{lrrrrrrrr}
\hline
Method & It. & Tot (s) & Avg res (s) & Step$_f$ & Avg in. & Max in. & \% max & Succ. (\%)\\
\hline
SPPA & 250 & 6.3182 & 0.02494 & 2.2660e+00 & 16.63 & 18 & 0.0 & 100.0 \\
WPPA & 250 & 4.8617 & 0.01935 & 3.3447e+00 & 16.04 & 18 & 0.0 & 100.0 \\
\hline
\end{tabular}
\caption{Computational performance for the image restoration experiment.}
\label{tab:image_gep_perf}
\end{table}

\begin{table}[!ht]
\centering
\begin{tabular}{lrrr}
\hline
Method & Constr. viol. & $\lVert x_n-\bar{x}\rVert$ & $R(x_n)$\\
\hline
SPPA & 0 & 1.7550e+03 & 7.6686e+00 \\
WPPA & 0 & 1.9282e+03 & 3.3581e+00 \\
\hline
\end{tabular}
\caption{Accuracy and feasibility metrics for the image restoration experiment.}
\label{tab:image_gep_acc}
\end{table}



In [26]:
# -----------------------------
# Replay helper + save images
# -----------------------------
import os
from matplotlib import pyplot as plt

def replay_method(x0, step_factory, maxit):
    """
    Recompute the trajectory of a method from x0 using the same step factory.
    Returns the full trajectory [x0, x1, ..., x_maxit].
    """
    x = x0.copy()
    traj = [x.copy()]
    step_fn = step_factory()
    for k in range(maxit):
        x, info = step_fn(x, k)
        traj.append(x.copy())
    return traj

def save_gray_image(arr2d, filename, title=None):
    """
    Save a grayscale image array to FIGDIR with fixed display range [0,255].
    """
    path = os.path.join(FIGDIR, filename)
    plt.figure(figsize=(5, 5))
    plt.imshow(arr2d, cmap="gray", vmin=0, vmax=255)
    if title is not None:
        plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    return path

# Recompute trajectories once, cleanly
traj_sppa = replay_method(x0, make_sppa_step, maxit)
traj_wppa = replay_method(x0, make_wppa_step, maxit)

x_final_sppa = traj_sppa[-1]
x_final_wppa = traj_wppa[-1]

# Reshape images
img_true = x_true.reshape(M, N)
img_obs  = proj_C(y_obs).reshape(M, N)
img_sppa = x_final_sppa.reshape(M, N)
img_wppa = x_final_wppa.reshape(M, N)

# Save individual images
p_true = save_gray_image(img_true, "image_gep_original.png", "Original image")
p_obs  = save_gray_image(img_obs,  "image_gep_observed.png", "Blurred + noisy image")
p_sppa = save_gray_image(img_sppa, "image_gep_sppa.png", "Recovered image (SPPA)")
p_wppa = save_gray_image(img_wppa, "image_gep_wppa.png", "Recovered image (WPPA)")

print("Saved:")
print(" -", p_true)
print(" -", p_obs)
print(" -", p_sppa)
print(" -", p_wppa)

Saved:
 - figures/image_gep_original.png
 - figures/image_gep_observed.png
 - figures/image_gep_sppa.png
 - figures/image_gep_wppa.png


In [27]:
# -----------------------------
# Image quality metrics + LaTeX table
# -----------------------------
import numpy as np
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

def compute_snr(x_ref, x_rec):
    num = np.linalg.norm(x_ref)
    den = np.linalg.norm(x_ref - x_rec)
    if den == 0:
        return np.inf
    return 20.0 * np.log10(num / den)

def compute_metrics(img_true, img_rec):
    snr = compute_snr(img_true, img_rec)
    psnr = peak_signal_noise_ratio(img_true, img_rec, data_range=255)
    ssim = structural_similarity(img_true, img_rec, data_range=255)
    return snr, psnr, ssim

# Compute metrics
snr_sppa, psnr_sppa, ssim_sppa = compute_metrics(img_true, img_sppa)
snr_wppa, psnr_wppa, ssim_wppa = compute_metrics(img_true, img_wppa)

# Print nicely
print("\nImage Quality Metrics:")
print(f"SPPA -> SNR: {snr_sppa:.4f} dB | PSNR: {psnr_sppa:.4f} dB | SSIM: {ssim_sppa:.4f}")
print(f"WPPA -> SNR: {snr_wppa:.4f} dB | PSNR: {psnr_wppa:.4f} dB | SSIM: {ssim_wppa:.4f}")


Image Quality Metrics:
SPPA -> SNR: 20.7007 dB | PSNR: 25.3893 dB | SSIM: 0.6802
WPPA -> SNR: 19.8835 dB | PSNR: 24.5721 dB | SSIM: 0.6006


In [28]:
# -----------------------------
# LaTeX table for paper
# -----------------------------
print("\nLaTeX table:\n")

print("\\begin{table}[!ht]")
print("\\centering")
print("\\begin{tabular}{lccc}")
print("\\hline")
print("Method & SNR (dB) & PSNR (dB) & SSIM\\\\")
print("\\hline")
print(f"SPPA & {snr_sppa:.4f} & {psnr_sppa:.4f} & {ssim_sppa:.4f}\\\\")
print(f"WPPA & {snr_wppa:.4f} & {psnr_wppa:.4f} & {ssim_wppa:.4f}\\\\")
print("\\hline")
print("\\end{tabular}")
print("\\caption{Image quality metrics for the restored images.}")
print("\\label{tab:image_gep_quality}")
print("\\end{table}")


LaTeX table:

\begin{table}[!ht]
\centering
\begin{tabular}{lccc}
\hline
Method & SNR (dB) & PSNR (dB) & SSIM\\
\hline
SPPA & 20.7007 & 25.3893 & 0.6802\\
WPPA & 19.8835 & 24.5721 & 0.6006\\
\hline
\end{tabular}
\caption{Image quality metrics for the restored images.}
\label{tab:image_gep_quality}
\end{table}
